# 🔮 Notebook 4 — Forecasting Next Month's Spending

We use **linear regression** on 3 months of data to predict April 2024 spending
per category, then compare the forecast against budget to flag risk areas early.

This is the kind of forward-looking analysis that turns data into action.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import plotly.express as px
import plotly.graph_objects as go

tx  = pd.read_csv('../data/cleaned.csv', parse_dates=['date'])
bud = pd.read_csv('../data/budgets.csv')
expenses = tx[tx['type']=='Expense'].copy()
print('Data loaded ✅')

## Step 1 — Prepare Monthly Category Spend

We need each category's spend for months 1, 2, and 3 as training data for the regression.

In [ ]:
cat_monthly = expenses.groupby(['month','category'])['amount'].sum().reset_index().rename(columns={'amount':'actual'})
print('Categories:', sorted(cat_monthly['category'].unique()))
cat_monthly.pivot(index='category', columns='month', values='actual').fillna(0).rename(columns={1:'January',2:'February',3:'March'})

## Step 2 — Linear Regression per Category

For each category we fit `spend = a × month + b` using months 1, 2, 3 as X and
actual spend as y. Then we predict month 4 (April).

**Why linear regression here?**
With only 3 data points it is the simplest model that can detect a trend direction.
It answers: is this category trending up, down, or staying flat?

In [ ]:
forecasts = []
for cat, group in cat_monthly.groupby('category'):
    group = group.sort_values('month')
    X = group['month'].values.reshape(-1, 1)
    y = group['actual'].values

    model = LinearRegression()
    model.fit(X, y)
    pred  = max(0, model.predict([[4]])[0])
    slope = model.coef_[0]
    trend = '↑ Increasing' if slope > 50 else ('↓ Decreasing' if slope < -50 else '→ Stable')

    forecasts.append({
        'category':     cat,
        'jan':          y[0] if len(y) > 0 else 0,
        'feb':          y[1] if len(y) > 1 else 0,
        'mar':          y[2] if len(y) > 2 else 0,
        'apr_forecast': round(pred, 2),
        'trend':        trend,
        'slope':        round(slope, 2)
    })

fdf = pd.DataFrame(forecasts).merge(bud, on='category', how='left')
fdf['vs_budget'] = fdf['apr_forecast'] - fdf['monthly_budget']
fdf['flag'] = fdf['vs_budget'].apply(lambda v: '⚠️ Over Budget' if v > 0 else '✅ Within Budget')

print('April 2024 Forecast:')
fdf[['category','jan','feb','mar','apr_forecast','trend','flag']].to_string(index=False)

In [ ]:
print(fdf[['category','jan','feb','mar','apr_forecast','trend','flag']].to_string(index=False))

## Step 3 — Visualise Forecast vs Budget

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Budget', x=fdf['category'], y=fdf['monthly_budget'],
                      marker_color='#555555'))
fig.add_trace(go.Bar(name='Apr Forecast', x=fdf['category'], y=fdf['apr_forecast'],
                      marker_color='#ff6b00'))
fig.update_layout(barmode='group', title='April 2024 Forecast vs Budget',
                  xaxis_title='Category', yaxis_title='Amount (R)',
                  template='plotly_dark', xaxis_tickangle=-30)
fig.show()

## Step 4 — Trend Chart per Category

Shows actual spend for Jan/Feb/Mar with the regression trend line extended to April.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cats = sorted(fdf['category'].unique())
fig = make_subplots(rows=4, cols=3,
                    subplot_titles=cats,
                    shared_yaxes=False)

for i, cat in enumerate(cats):
    row = i // 3 + 1
    col = i % 3 + 1
    grp = cat_monthly[cat_monthly['category']==cat].sort_values('month')
    X = grp['month'].values.reshape(-1,1)
    y = grp['actual'].values
    model = LinearRegression()
    model.fit(X, y)
    x_line = [1, 2, 3, 4]
    y_line  = [max(0, model.predict([[m]])[0]) for m in x_line]

    fig.add_trace(go.Scatter(x=[1,2,3], y=y, mode='markers+lines',
                             name=cat, marker=dict(color='#ff6b00', size=8),
                             showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines',
                             line=dict(dash='dash', color='white'),
                             showlegend=False), row=row, col=col)

fig.update_layout(height=900, title='Spending Trend per Category (dashed = forecast)',
                  template='plotly_dark')
fig.show()

## Step 5 — Forecast Summary

In [ ]:
total_forecast = fdf['apr_forecast'].sum()
total_budget   = fdf['monthly_budget'].sum()
over_cats = fdf[fdf['vs_budget'] > 0]

print('='*55)
print('APRIL 2024 FORECAST SUMMARY')
print('='*55)
print(f'Total forecast spend : R {total_forecast:,.2f}')
print(f'Total budget         : R {total_budget:,.2f}')
print(f'Variance             : R {total_forecast - total_budget:+,.2f}')
print()
print(f'Categories forecast over budget ({len(over_cats)}):')
for _, row in over_cats.iterrows():
    print(f'  ⚠️  {row["category"]:<18} forecast R {row["apr_forecast"]:,.0f}  '
          f'(budget R {row["monthly_budget"]:,.0f}, +R {row["vs_budget"]:,.0f})')
print('='*55)